In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
df = pd.read_excel('Google.xlsx')

In [3]:
df['MRP'] = df['MRP'].str.replace('₹', '').str.replace(' ', '').str.replace(',', '').astype(float)
df['Current Price'] = df['Current Price'].str.replace('₹', '').str.replace(',', '').astype(float)

In [4]:
def extract_discount(value):
    if pd.isna(value):
        return 0  # or return None
    return int(value.replace('% off', ''))

df['Discount_Number'] = df['Discount'].apply(extract_discount)

In [5]:
df.drop("Discount", axis=1, inplace=True)

In [6]:
brands = {
    'Apple': ['apple', 'iphone', 'ipad'],
    'motorola': ['motorola', 'moto'],
    'AI_plus': ['ai plus', 'aiplus'],
    'IQOO': ['iqoo'],
    'Tecno': ['tecno'],
    'lava': ['lava'],
    'readmi': ['readmi', 'redmi'],  
    'realme': ['realme'],
    'POCO': ['poco'],
    'nothing': ['nothing'],
    'Infix': ['infix'],
    'Oppo': ['oppo'],
    'vivo': ['vivo'],
    'Google': ['google', 'pixel'],
    'OnePlus': ['oneplus', 'one plus'],
    'Samsung': ['samsung', 'galaxy'],
}

def get_brand(product_name):
    if pd.isna(product_name):
        return 'Unknown'
    
    product_name = str(product_name).lower()
    
    for brand, keywords in brands.items():
        for keyword in keywords:
            if keyword in product_name:
                return brand
    return 'Unknown'
df['Brand'] = df['Product Name'].apply(get_brand)

In [7]:
df['RAM'] = df['RAM'].str.extract(r'(\d+)').astype(int)

In [8]:
df['Storage'] = df['Storage'].str.extract(r'(\d+)').astype(int)

In [9]:
def extract_ratings_reviews(text):
    """
    Extract ratings and reviews from text
    Example: "2,46,375 Ratings & 9,265 Reviews"
    Returns: (ratings, reviews)
    """
    if pd.isna(text):
        return None, None
    
    text = str(text)
    
    # Extract ratings (numbers before 'Ratings')
    ratings_match = re.search(r'([\d,]+)\s*Ratings', text, re.IGNORECASE)
    ratings = ratings_match.group(1).replace(',', '') if ratings_match else None
    
    # Extract reviews (numbers before 'Reviews')
    reviews_match = re.search(r'([\d,]+)\s*Reviews', text, re.IGNORECASE)
    reviews = reviews_match.group(1).replace(',', '') if reviews_match else None
    
    return ratings, reviews

# Apply to dataframe
df[['Ratings', 'Reviews']] = df['Review text'].apply(
    lambda x: pd.Series(extract_ratings_reviews(x))
)

# Convert to numbers
df['Ratings'] = pd.to_numeric(df['Ratings'], errors='coerce')
df['Reviews'] = pd.to_numeric(df['Reviews'], errors='coerce')

In [10]:
def extract_battery(text):
    if pd.isna(text):
        return 0
    match = re.search(r'(\d+)', str(text))
    return int(match.group()) if match else 0

df['Battery'] = df['Battery'].apply(extract_battery)

In [11]:
df.head()

,Product Name,Current Price,MRP,Rating,Review text,RAM,Storage,Display,Camera,Battery,Processor,Product URL,Image URL,Discount_Number,Brand,Ratings,Reviews
0,"Google Pixel 10a (Lavender, 256 GB)",49999.0,NaN,4.5,"1,646 Ratings & 164 Reviews",8,256,16.0 cm (6.3 inch) Full HD+ AMOLED Display,48MP + 13MP | 13MP Front Camera,5100,Tensor G4 Processor,https://www.flipkart.com/google-pixel-10a-lave...,https://rukminim2.flixcart.com/image/312/312/x...,0,Google,1646,164
1,"Google Pixel 9A (Obsidian, 256 GB)",39999.0,49999.0,4.4,"10,995 Ratings & 812 Reviews",8,256,15.96 cm (6.285 inch) Full HD+ Display,48MP + 12MP | 13MP Front Camera,5100,Tensor G4 Processor,https://www.flipkart.com/google-pixel-9a-obsid...,https://rukminim2.flixcart.com/image/312/312/x...,20,Google,10995,812
2,"Google Pixel 10a (Obsidian, 256 GB)",49999.0,NaN,4.5,"1,646 Ratings & 164 Reviews",8,256,16.0 cm (6.3 inch) Full HD+ AMOLED Display,48MP + 13MP | 13MP Front Camera,5100,Tensor G4 Processor,https://www.flipkart.com/google-pixel-10a-obsi...,https://rukminim2.flixcart.com/image/312/312/x...,0,Google,1646,164
3,"Google Pixel 10a (Berry, 256 GB)",49999.0,NaN,4.5,"1,646 Ratings & 164 Reviews",8,256,16.0 cm (6.3 inch) Full HD+ AMOLED Display,48MP + 13MP | 13MP Front Camera,5100,Tensor G4 Processor,https://www.flipkart.com/google-pixel-10a-berr...,https://rukminim2.flixcart.com/image/312/312/x...,0,Google,1646,164
4,"Google Pixel 10a (Fog, 256 GB)",49999.0,NaN,4.5,"1,646 Ratings & 164 Reviews",8,256,16.0 cm (6.3 inch) Full HD+ AMOLED Display,48MP + 13MP | 13MP Front Camera,5100,Tensor G4 Processor,https://www.flipkart.com/google-pixel-10a-fog-...,https://rukminim2.flixcart.com/image/312/312/x...,0,Google,1646,164


In [12]:
df.isnull().sum()

Product Name        0
Current Price       0
MRP                19
Rating              0
Review text         0
RAM                 0
Storage             0
Display             0
Camera              0
Battery             0
Processor           1
Product URL         0
Image URL           0
Discount_Number     0
Brand               0
Ratings             0
Reviews             0
dtype: int64

In [13]:
df['MRP'] = np.where(
    (df['Discount_Number'] == 0) & (df['MRP'].isna() | (df['MRP'] == '')),
    df['Current Price'],
    df['MRP']
)

In [14]:
df.isnull().sum()

Product Name       0
Current Price      0
MRP                0
Rating             0
Review text        0
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          1
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
dtype: int64

In [15]:
print(df[df['Processor'].isnull()])

                     Product Name  Current Price      MRP  Rating  \
18  Google GP 3XL (White, 128 GB)        21216.0  25799.0     3.0   

              Review text  RAM  Storage                     Display  \
18  2 Ratings & 1 Reviews    4      128  16.0 cm (6.3 inch) Display   

                Camera  Battery Processor  \
18  12.2MP Rear Camera     5000       NaN   

                                          Product URL  \
18  https://www.flipkart.com/google-gp-3xl-white-1...   

                                            Image URL  Discount_Number  \
18  https://rukminim2.flixcart.com/image/312/312/x...               17   

     Brand  Ratings  Reviews  
18  Google        2        1  


In [16]:
df.loc[df['Processor'].isnull(), 'Processor'] = 'Qualcomm Snapdragon 845'

In [17]:
df.isnull().sum()

Product Name       0
Current Price      0
MRP                0
Rating             0
Review text        0
RAM                0
Storage            0
Display            0
Camera             0
Battery            0
Processor          0
Product URL        0
Image URL          0
Discount_Number    0
Brand              0
Ratings            0
Reviews            0
dtype: int64

In [18]:
df.to_excel('Cleaned_Google.xlsx', index=False)